In [5]:
#7
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dataset
data = {
    "Movie": ["Inception", "Interstellar", "The Dark Knight", "Memento", "The Prestige"],
    "Description": [
        "dreams mind-bending science-fiction thriller",
        "space travel black holes emotional sci-fi drama",
        "vigilante hero action crime thriller",
        "memory loss psychological thriller mystery",
        "magicians rivalry mystery psychological thriller"
    ]
}

df = pd.DataFrame(data)

# TF-IDF computation
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["Description"])

# Similarity matrix
similarity = cosine_similarity(tfidf_matrix)

# Choose target movie
target = 3   # Memento

# Print recommended movies (sorted by similarity)
print("Recommended movies for:", df["Movie"][target])
scores = similarity[target]

# Sort, skip the target movie itself, show others
recommended = df.iloc[scores.argsort()[::-1][1:]]

print(recommended)



Recommended movies for: Memento
             Movie                                       Description
4     The Prestige  magicians rivalry mystery psychological thriller
2  The Dark Knight              vigilante hero action crime thriller
0        Inception      dreams mind-bending science-fiction thriller
1     Interstellar   space travel black holes emotional sci-fi drama


In [6]:
#8q
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Dataset
data = {
    "User": ["U1", "U2", "U3", "U4", "U5"],
    "Item1": [5, 4, 0, 2, 0],
    "Item2": [3, 0, 4, 0, 2],
    "Item3": [4, 0, 0, 3, 0],
    "Item4": [0, 2, 5, 0, 4]
}

df = pd.DataFrame(data).set_index("User")

# Item-item similarity
similarity = cosine_similarity(df.T)

item_sim_df = pd.DataFrame(
    similarity,
    index=df.columns,
    columns=df.columns
)

# Choose target item
target_item = "Item1"

print("Item-based similar items for:", target_item)

# Similarity scores
scores = item_sim_df[target_item]

# Sort (highest to lowest), skip itself
recommended = scores.sort_values(ascending=False)[1:]

print(recommended)


Item-based similar items for: Item1
Item3    0.775170
Item2    0.415227
Item4    0.177778
Name: Item1, dtype: float64


In [7]:
#q9 model based
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

# Data
data = {
    "User": ["U1","U2","U3","U4","U5"],
    "Item1": [5,4,0,2,3],
    "Item2": [3,0,4,0,2],
    "Item3": [4,0,0,3,0],
    "Item4": [0,2,5,0,4]
}

df = pd.DataFrame(data).set_index("User")

# SVD model
svd = TruncatedSVD(n_components=2)
U = svd.fit_transform(df)       # user latent features
V = svd.components_             # item latent features

# Predicted ratings
pred = U @ V
pred_df = pd.DataFrame(pred, index=df.index, columns=df.columns)

# Recommend for a user
target = "U3"
print("Predicted ratings for", target)
print(pred_df.loc[target].sort_values(ascending=False))


Predicted ratings for U3
Item4    5.210363
Item2    3.158753
Item1    0.868854
Item3   -0.897951
Name: U3, dtype: float64


In [9]:
#q10 user based recommendation system 
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Rating matrix
data = {
    "User": ["U1","U2","U3","U4","U5"],
    "Item1": [5,4,0,2,3],
    "Item2": [3,0,4,0,2],
    "Item3": [4,0,0,3,0],
    "Item4": [0,2,5,0,4]
}

df = pd.DataFrame(data).set_index("User")

# 1. Compute cosine similarity between users
user_sim = pd.DataFrame(
    cosine_similarity(df),
    index=df.index,
    columns=df.index
)

print("User Similarity Matrix:\n", user_sim)

# 2. Predict rating for a target item
target_item = "Item3"

# Ratings for that item
item_ratings = df[target_item]

# Weighted prediction using user similarity
pred = user_sim.dot(item_ratings) / user_sim.sum(axis=1)

pred_df = pd.DataFrame(pred, columns=[f"Predicted_{target_item}"])
print("\nPredicted Ratings:\n", pred_df)

# 3. Sort predictions
print("\nTop User Recommendations:\n")
print(pred_df.sort_values(by=f"Predicted_{target_item}", ascending=False))


User Similarity Matrix:
 User        U1        U2        U3        U4        U5
User                                                  
U1    1.000000  0.632456  0.265036  0.862911  0.551487
U2    0.632456  1.000000  0.349215  0.496139  0.830455
U3    0.265036  0.349215  1.000000  0.000000  0.812021
U4    0.862911  0.496139  0.000000  1.000000  0.309016
U5    0.551487  0.830455  0.812021  0.309016  1.000000

Predicted Ratings:
       Predicted_Item3
User                 
U1           1.989418
U2           1.214606
U3           0.436943
U4           2.418098
U5           0.894380

Top User Recommendations:

      Predicted_Item3
User                 
U4           2.418098
U1           1.989418
U2           1.214606
U5           0.894380
U3           0.436943
